# World-model planning: raw versus multi-goal ranking cost

Select **GPU → Run all**. Evaluate the frozen **multi-goal ranking adapter** from your latest run against raw latent MSE, using the same frozen world model and CEM planner. **No adapter training occurs.**

Default: **6 goals × 8 starts = 48 paired tasks, 96 episodes**. Goals: **(50,42), (46,38), (40,18), (50,18), (56,10), (50,8)**. Goal 0 is the seen original reference; the others were withheld from adapter fitting.

Both endpoints stay inside the original **R34 disk**, on the **right of the wall**. Starts use x38..56/y8..55, with 2 near, 3 medium and 3 long starts per goal, at distances 6..10,10..18,18..30. Same start/goal/reset/CEM seed for both costs.

The notebook times one matched pair first, then reuses it when completing the full benchmark. This includes substantially more planning than the earlier static heatmap tests. Every completed episode is saved for resumption. Earlier experiments/results remain unchanged.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
SOURCE_RUN=DRIVE/'datasets/curriculum_cost_adapter/radial_multigoal_ranking_v1/20261106T195304628603Z'
assert (SOURCE_RUN/'branches/multi/complete.json').is_file(), 'Set SOURCE_RUN to your completed multi-goal ranking run.'
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/ranking_planning_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-ranking-planning')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='PUBLISH_SOURCE_COMMIT'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/ranking-planning-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'
VIDEOS=False

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Frozen ranking source:',SOURCE_RUN)
print('New planning results:',OUTPUT)

## Load the pinned code and frozen artifacts

Use the same matched-v3 fixed-layout world model as the earlier planning experiments. Hashes verify the exact encoder and world model. Historical ranking/source code fingerprints are recorded; this read-only evaluation does not require old local caches.

The environment layout stays `(32,30)`. Encoder images contain no goal marker; optional videos add a goal marker only for viewing.

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

def command(operation,condition=None,limit=None):
    if operation not in ('prepare','goals','evaluate','report'):raise ValueError('Unknown operation.')
    if operation=='evaluate' and condition not in ('raw','adapted'):raise ValueError('Choose raw or adapted condition.')
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.ranking_planning',operation,
          '--source-run',SOURCE_RUN,'--output-dir',OUTPUT]
    if operation in ('goals','evaluate'):
        args+=['--world-model-checkpoint',CHECKPOINT,'--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--device','cuda']
        if VIDEOS:args+=['--videos']
    if operation=='evaluate':
        args+=['--condition',condition]
        if limit is not None:args+=['--limit',limit]
    return args

print('Ready. Model/encoder/adapter weights stay frozen throughout evaluation.')

## Fix the paired start–goal cases before planning

The cases are chosen from geometry only, before either planner runs. Starts stay at least 0.5 units from fitting nodes, reference positions and other starts. Pixel aliases are checked before encoding references. Straight paths between the endpoints remain on the fitted side of the wall.

There is no easier-case selection based on success. The manifest contains every reset/CEM seed and exact coordinate.

In [ ]:
shell(command('prepare'),cwd=REPO)
manifest=json.loads((OUTPUT/'manifest.json').read_text())
config=json.loads((OUTPUT/'experiment.json').read_text())
print('Frozen ranking checkpoint:',config['source']['checkpoint_files']['adapter.pt'])
print('Planner:',manifest['planning'])
print('Goals:',manifest['goals'])
print('Tasks / total episodes:',len(manifest['evaluation']),2*len(manifest['evaluation']))
for case in manifest['evaluation']:
    print(case['case_id'],'goal=',case['goal'],'start=',[round(v,3) for v in case['start']],case['band'],'seed=',case['seed'])

## Encode a separate shared reference for each goal

Fresh marker-free goal images are encoded once in float32 and saved. Raw and adapted costs use these same reference tensors. Every case verifies that its reset renders the intended goal image; the old fixed-goal cache is not used.

Original cached/fresh consistency is printed as a diagnostic. Reference and planning runtime signatures must match, including GPU name/precision, renderer and encoder.

In [ ]:
started=time.monotonic()
shell(command('goals'),cwd=REPO)
print('Goal-reference preparation minutes:',round((time.monotonic()-started)/60,2))
print('Consistency:',json.loads((OUTPUT/'reference/consistency.json').read_text()))

## Time one matched pair

Run the first task with both costs. These two completed episodes will be reused in the full benchmark. Timing includes process/model startup and is not a precise forecast for every start–goal distance.

The planner uses horizon 5 macros, executes up to 5 before replanning, population 300, 30 elites, 10 CEM iterations, primitive action norm cap 3.5, and at most 20 macros (100 primitive actions). Same random seeds; candidate values may diverge as the costs select different elites.

In [ ]:
started=time.monotonic()
for condition in ('raw','adapted'):shell(command('evaluate',condition,limit=1),cwd=REPO)
print(f'Initial matched pair wall time: {(time.monotonic()-started)/60:.2f} minutes (including startup).')
print('Unique episodes remaining:',2*len(manifest['evaluation'])-2)
print('Completed episodes are reused when continuing below.')

## Complete both planning conditions

Only the terminal latent cost changes. World-model histories remain raw, and proprioception still conditions the same frozen dynamics model. No position oracle or region penalty is added to either planner.

Endpoints are constrained; executed paths may wander. Every primitive position/action is saved, and departures from the original fitting region are reported. Failures and excursions stay in the denominator. Success remains environment distance **strictly below 4.5**.

Each completed episode is committed immediately. If interrupted, rerun this cell in the same runtime to resume missing cases. A changed GPU/software/configuration requires a fresh run timestamp rather than mixing conditions.

In [ ]:
started=time.monotonic()
for condition in ('raw','adapted'):
    print('EVALUATING',condition,flush=True)
    shell(command('evaluate',condition),cwd=REPO)
print(f'Remaining planning cell wall time: {(time.monotonic()-started)/60:.2f} minutes.')

## Paired outcomes and efficiency

**Wins**: adapted succeeds where raw fails. **Losses**: raw succeeds where adapted fails. Compare per goal and distance range; goal 0 is seen, other references were withheld from fitting.

Final distance includes all cases. Efficiency uses only the cases where **both methods succeed**, so stopping unsuccessfully does not appear efficient. Region-exit counts and in-region successes are shown separately. These are descriptive results on fixed, correlated tasks.

In [ ]:
shell(command('report'),cwd=REPO)
r=json.loads((OUTPUT/'report.json').read_text());manifest=r['manifest']
from IPython.display import display,Image

def number(v):return 'N/A' if v is None else f'{v:.3f}'

def print_summary(label,row):
    print(label,'| episodes:',row['episodes'],'| wins/losses:',row['wins'],row['losses'],'| both succeed/fail:',row['both_success'],row['both_fail'])
    for condition in ('raw','adapted'):
        c=row['conditions'][condition]
        print(' ',condition,'success:',c['successes'],'/',row['episodes'],
              '| final distance:',number(c['mean_final_distance']),'| exited fitting region:',c['excursion_episodes'],
              '| in-region successes:',c['in_region_successes'])
    print('  Both-success efficiency:',row['both_success_efficiency'])

print('Source/checkpoints unchanged:',r['references_unchanged'])
print('Shared goal consistency:',r['reference_consistency'])
print_summary('ALL GOALS',r['summary']['overall'])
print_summary('WITHHELD GOALS ONLY',r['summary']['withheld'])
for gi,g in r['summary']['by_goal'].items():print_summary(f'GOAL {gi} {g["goal"]} ({g["status"]})',g)
for band,row in r['summary']['by_distance_band'].items():print_summary('START DISTANCE: '+band,row)
print('Interaction journals (include interrupted/repeated attempts):',json.dumps(r['journals'],indent=2))
for name in ('case_coverage','success_comparison','paired_outcomes'):display(Image(filename=str(OUTPUT/f'{name}.png')))
print('Interpretation:',r['summary']['note'])

## Paired trajectories for every goal

Each panel shows one shared start–goal task: gray is raw cost, orange is ranking-adapted cost. The full primitive paths are plotted. The orange dotted circle is the success radius; the blue dashed curve is the fitting-region boundary; shading shows the wall.

Limits expand when needed to show excursions. Read each panel's success/steps and region-exit labels together. Cost values themselves are not compared across raw and adapted units.

In [ ]:
for gi,goal in enumerate(manifest['goals']):
    print('GOAL',gi,goal)
    display(Image(filename=str(OUTPUT/f'goal_{gi}_trajectories.png')))
raw={item['case_id']:item for item in r['records']['raw']}
adapted={item['case_id']:item for item in r['records']['adapted']}
print('PER-CASE: success, primitive steps, final distance, region exit')
for case in manifest['evaluation']:
    print(case['case_id'],{condition:dict(success=rows[case['case_id']]['success'],steps=rows[case['case_id']]['micro_steps'],
          final_distance=rows[case['case_id']]['final_goal_distance'],exit=rows[case['case_id']]['ever_outside_fitting'])
          for condition,rows in [('raw',raw),('adapted',adapted)]})
print('Results:',OUTPUT)

## Return the executed notebook

Download **File → Download → Download .ipynb** and attach it here. Drive retains the fixed manifest, shared references, runtime/model/source signatures, every episode, primitive trajectories/actions, attempt journal, paired summaries and PNG/PDF figures. Default videos are off; set `VIDEOS=True` in setup for a fresh run if wanted.

The adapter is frozen. Planning episodes do not feed fitting or change the source checkpoint. Local verification uses synthetic dynamics and real CEM/control-flow tests; actual GPU results establish planning performance for these tasks.